# DoubleML DR-DiD pre-trend test — PT_hold

**Workstream PT · the conditional-parallel-trends diagnostic (Reviewer 3.2.3)**

conditional PT holds: false-positive rate of the diagnostic (this is B1_baseline's DGP)

The benchmark counterpart of `Pretrend/Pretrend_PT_hold_lin_*.ipynb`: the same
question, the **same seeded panels**, the same estimands and the same summary
schema, so `did_bcf_revision.metrics.compute_metrics` scores DiD-BCF's Bayesian
diagnostic and this one on identical definitions.

**What the test is.** `att_gt(..., base_period = "universal")` reports, for every
pre-treatment period `t < g-1`,

```
ATT(g,t) = E[Y_t - Y_{g-1} | G=g] - E[Y_t - Y_{g-1} | never treated]
```

doubly-robustly **conditional on X**, with random-forest nuisances plugged in as
a custom `est_method` (Chang 2020). That is exactly `Delta(k)` of
`did_bcf_revision/pretrend.py` at `k = t - g`, verified against the DGP's
realised violation slope. So this is the **ML-conditional** pre-trend test — the
closest frequentist analogue of the Bayesian diagnostic in the suite.

Emitted: `Delta(k)` per pre-period, the implied differential **`slope`**, the
any-`k` decision rules (`any`, `any_bonf`), and `joint` — `did`'s own Wald
pre-test, halved onto the one-sided scale the metrics layer reads.

> **No subgroup contrast (`PRE_SUBC`).** It needs four extra fits per
> replication and this is by far the most expensive estimator here.
> Callaway--Sant'Anna, did2s and grf-DiD all carry the sample-split contrast, so
> the heterogeneous-violation axis is covered.

> ⚠️ **Before running:** the clone must contain the `PT_*` scenarios in
> `did_bcf_revision/config.py`. If the setup cell raises
> `KeyError: No experiment named 'PT_hold'`, push the engine first. The R script
> itself is embedded below, so nothing else needs pushing.

> **Colab:** upload just this notebook and *Run all*. Measured locally at
> ~14 s/replication with `NUM_TREES=500`, i.e. ~1.6 h for 200 reps x 2 linearity
> degrees; budget 2-4x that on a Colab CPU VM. Set `NUM_TREES=100` for a ~4x
> speed-up — the forest is only a nuisance learner, so the doubly-robust
> estimate barely moves.

In [ ]:
# ===== Parameters (edit me) =====
SCENARIO  = "PT_hold"
REPS      = 200     # replications per linearity degree (matches the DiD-BCF runs)
NUM_TREES = 500     # ranger trees for the DoubleML nuisances (100 ~ 4x faster)

# ===== Install R + the DoubleML stack as fast *binary* packages =====
import shutil, subprocess, os
if shutil.which('Rscript') is None:
    subprocess.run('sudo apt-get -qq update && sudo apt-get -qq install -y r-base',
                   shell=True, check=True)
codename = (subprocess.run(['bash','-lc','. /etc/os-release && echo $VERSION_CODENAME'],
            capture_output=True, text=True).stdout.strip() or 'jammy')
r_install = r'''
options(repos = c(CRAN = sprintf('https://packagemanager.posit.co/cran/__linux__/%s/latest', Sys.getenv('PPM_CODENAME'))),
        HTTPUserAgent = sprintf('R/%s R (%s)', getRversion(),
            paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
pkgs <- c('did','DoubleML','mlr3','mlr3learners','ranger','lgr','progress')
need <- pkgs[!pkgs %in% rownames(installed.packages())]
if (length(need)) install.packages(need)
cat('R packages ready:', paste(pkgs[pkgs %in% rownames(installed.packages())], collapse=', '), '\n')
'''
res = subprocess.run(['Rscript','-e', r_install], capture_output=True, text=True,
                     env={**os.environ, 'PPM_CODENAME': codename})
print(res.stdout[-3000:]); print(res.stderr[-1500:])

In [ ]:
# ===== Clone the engine + regenerate the seeded panels =====
# The panels are generated INLINE rather than through DGPs/data_creation_<scen>.py
# so this notebook does not depend on that script having been pushed, and so the
# `pt_slope` truth column is present even if the clone's exports.py predates it.
import os, glob, subprocess, sys
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
if not os.path.isdir("DiD-BCF"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
ROOT = "DiD-BCF/Simulation_Studies_Revision"
%pip install -q numpy pandas
sys.path.insert(0, ROOT)

from did_bcf_revision.config import get_experiment
from did_bcf_revision.dgps import generate_canonical_did
from did_bcf_revision.exports import to_r_frame
try:                                   # PT runs degrees (1, 3) only
    from did_bcf_revision.config import degrees_for
    DEGREES = degrees_for("PT")
except ImportError:
    DEGREES = (1, 3)

try:
    exp = get_experiment(SCENARIO)
except KeyError as exc:
    raise SystemExit(
        f"{exc}\n\nThe clone's did_bcf_revision/config.py has no PT_* scenarios. "
        "Commit and push the engine, then re-run this cell.") from None

folder = f"{ROOT}/R_code/{SCENARIO}_datasets"
os.makedirs(folder, exist_ok=True)
FOLDER = os.path.abspath(folder)
for d in DEGREES:
    outdir = f"{folder}/linearity_degree={d}"
    os.makedirs(outdir, exist_ok=True)
    for rep in range(REPS):
        df = generate_canonical_did(seed=int(rep),
                                    **{**exp.dgp_params, "n_units": int(exp.n_values[0]),
                                       "linearity_degree": int(d)})
        r = to_r_frame(df)
        if "pt_slope" not in r.columns:        # clone predates the truth column
            r["pt_slope"] = df["pt_slope"].to_numpy()
        # Zero-padded so an alphabetical list.files() is also numeric order.
        r.to_csv(f"{outdir}/iteration_{rep:04d}.csv", index=False)
print("panels ready for", SCENARIO, "| degrees:", DEGREES, "| reps:", REPS)
print("  ->", FOLDER)

## The method script

A copy of `R_code/PT_*_datasets/DoubleML_pretrend.R`, embedded so the notebook is
self-contained and does not depend on that file having been pushed. The next
cell writes it into the scenario folder.

In [ ]:
# ===== Write the method script into the scenario folder =====
# SETTING is rewritten from the SCENARIO set above, so editing that one
# parameter is enough -- the script cannot end up labelling its output
# with a different scenario than the panels it just read.
import re
DOUBLEML_PRETREND_R = r"""
# DoubleML doubly-robust PRE-TREND test (random-forest nuisances) for scenario
# "PT_hold".
#
# The benchmark counterpart of did_bcf_revision/pretrend.py, emitting the SAME
# estimands in the SAME schema (method = "doubleml") on the SAME seeded panels.
# Identical to did_dr_pretrend.R except that Callaway--Sant'Anna's `att_gt` is
# plugged with DoubleML's ATTE estimator (Chang 2020) instead of the analytical
# doubly-robust one, so the nuisances are random forests rather than
# logit/OLS -- this is the ML-conditional pre-trend test, the closest
# frequentist analogue of the Bayesian diagnostic.
#
# `base_period = "universal"` makes ATT(g,t) at t < g-1 exactly Delta(k) of
# pretrend.py; see did_dr_pretrend.R for the estimand and the inference objects.
#
# **No subgroup contrast (PRE_SUBC).** The contrast needs four extra fits per
# replication, and this estimator costs ~20-90 s per fit against ~0.15 s for
# Callaway--Sant'Anna, which is why it lives on Colab at all. Callaway--Sant'Anna
# and grf-DiD carry the sample-split contrast for the frequentist side.
library(did)
library(progress)
library(DoubleML)
library(mlr3)
library(mlr3learners)
library(lgr)
lgr::get_logger("mlr3")$set_threshold("fatal")

# Plug DoubleML's ATTE estimator into att_gt as a custom est_method (Chang 2020).
# `att_gt` hands it (y1, y0) = the outcome pair for the (g,t) cell being
# estimated, which under base_period="universal" is (Y_t, Y_{g-1}) for the
# pre-treatment placebos too, so no change is needed for the pre-trend use.
doubleml_did_rf <- function(y1, y0, D, covariates,
                            ml_g = lrn("regr.ranger", num.trees = 500),
                            ml_m = lrn("classif.ranger", num.trees = 500),
                            n_folds = 5, n_rep = 1, ...) {
  delta_y <- y1 - y0
  dml_data <- DoubleML::double_ml_data_from_matrix(X = covariates, y = delta_y, d = D)
  dml_obj <- DoubleML::DoubleMLIRM$new(dml_data, ml_g = ml_g, ml_m = ml_m,
                                       score = "ATTE", n_folds = n_folds)
  dml_obj$fit()
  list(ATT = dml_obj$coef[1], att.inf.func = dml_obj$psi[, 1, 1])
}

sink("output_DoubleML_pretrend.txt")
DGP <- "canonical"; SETTING <- "PT_hold"; METHOD <- "doubleml"
REF_K <- -1L

# ---- DiD-BCF-schema summary emission (shared helpers) -----------------------
SCHEMA <- c("dgp","setting","linearity_degree","N","rep","estimand_type",
            "estimand_id","g","t","k","method","post_mean","sd","q025","q05",
            "q95","q975","p_bayes","surf_rmse","surf_mae","surf_n","surf_mape",
            "surf_cover95","surf_len95","surf_cover90","surf_len90","true")
Z95 <- 1.959964; Z90 <- 1.644854
wald_tail <- function(est, se) if (is.finite(est) && is.finite(se) && se > 0)
  pnorm(-abs(est / se)) else NA_real_

new_scalar <- function(estimand_type, estimand_id, g, t, k, est, se, truth) {
  data.frame(estimand_type = estimand_type, estimand_id = estimand_id,
             g = g, t = t, k = k, post_mean = est, sd = se,
             q025 = est - Z95 * se, q05 = est - Z90 * se,
             q95 = est + Z90 * se, q975 = est + Z95 * se,
             p_bayes = wald_tail(est, se),
             surf_rmse = NA_real_, surf_mae = NA_real_, surf_n = NA_integer_,
             surf_mape = NA_real_, surf_cover95 = NA_real_, surf_len95 = NA_real_,
             surf_cover90 = NA_real_, surf_len90 = NA_real_, true = truth,
             stringsAsFactors = FALSE)
}

# A per-replication DECISION rule (any-k, Bonferroni any-k, joint Wald).  It has
# no point estimate and no interval by construction; metrics.py routes these to
# the tail-only aggregator, which turns `true == 0` into a size and `true != 0`
# into a detection rate.
new_decision <- function(estimand_type, estimand_id, p, truth) {
  data.frame(estimand_type = estimand_type, estimand_id = estimand_id,
             g = NA_real_, t = NA_real_, k = NA_real_,
             post_mean = NA_real_, sd = NA_real_, q025 = NA_real_, q05 = NA_real_,
             q95 = NA_real_, q975 = NA_real_, p_bayes = p,
             surf_rmse = NA_real_, surf_mae = NA_real_, surf_n = NA_integer_,
             surf_mape = NA_real_, surf_cover95 = NA_real_, surf_len95 = NA_real_,
             surf_cover90 = NA_real_, surf_len90 = NA_real_, true = truth,
             stringsAsFactors = FALSE)
}

finalize <- function(rows, rep, N, dgp, setting, lin_degree, method) {
  if (!length(rows)) return(NULL)
  df <- do.call(rbind, rows)
  df$dgp <- dgp; df$setting <- setting; df$linearity_degree <- lin_degree
  df$N <- N; df$rep <- rep; df$method <- method
  df[, SCHEMA]
}

# ---- Truth ------------------------------------------------------------------
# Mirrors did_bcf_revision.pretrend.true_pretrend: the treated-minus-control
# difference in the per-unit violation slope `pt_slope` (a truth column carried
# by the exported panel, like CATE).  Delta(k) = diff * (k - REF_K); the `slope`
# estimand and every decision rule carry `diff` itself, which is what makes
# reject05 a size when the violation is zero and a detection rate otherwise.
# The X_2 subgroup boundaries are computed on the TREATED units and applied to
# the controls, so both arms are compared over the same region of X -- exactly
# pretrend._sel_on.
truth_of <- function(d) {
  u <- d[!duplicated(d$unit_id), ]
  ever <- u$eventually_treated == 1
  if (all(ever) || !any(ever)) return(NULL)
  ut <- u[ever, ]
  q1 <- as.numeric(quantile(ut$X_2, 1 / 3)); q2 <- as.numeric(quantile(ut$X_2, 2 / 3))
  sel <- list("X1=0" = u$X_1 <= 0.5, "X1=1" = u$X_1 > 0.5,
              "X2=low" = u$X_2 <= q1, "X2=high" = u$X_2 >= q2)
  sub <- vapply(sel, function(m) {
    if (sum(m & ever) < 5 || sum(m & !ever) < 5) return(NA_real_)
    mean(u$pt_slope[m & ever]) - mean(u$pt_slope[m & !ever])
  }, numeric(1))
  list(diff = mean(u$pt_slope[ever]) - mean(u$pt_slope[!ever]),
       contrast = c(X1 = sub[["X1=1"]] - sub[["X1=0"]],
                    X2 = sub[["X2=high"]] - sub[["X2=low"]]),
       cut = c(q1, q2))
}

# ---- The pre-trend block, shared by the aggregate test and the contrasts -----
# Emits per-k Delta(k), the implied differential slope, and the two any-k
# decision rules, in the estimand ids pretrend.py uses.
pre_rows <- function(kk, est, se, V, truth_slope, type = "PRE", prefix = "") {
  rows <- list()
  for (i in seq_along(kk)) {
    rows[[length(rows) + 1]] <- new_scalar(
      type, sprintf("%sk=%d", prefix, kk[i]), NA_real_, NA_real_,
      as.integer(kk[i]), est[i], se[i], truth_slope * (kk[i] - REF_K))
  }
  w <- kk - REF_K
  slope <- sum(w * est) / sum(w * w)
  slope_se <- if (!is.null(V))
    sqrt(max(as.numeric(t(w) %*% V %*% w), 0)) / sum(w * w)
  else sqrt(sum((w * se) ^ 2)) / sum(w * w)
  rows[[length(rows) + 1]] <- new_scalar(type, paste0(prefix, "slope"),
    NA_real_, NA_real_, NA_real_, slope, slope_se, truth_slope)
  tails <- mapply(wald_tail, est, se)
  p_min <- if (any(is.finite(tails))) min(tails, na.rm = TRUE) else NA_real_
  rows[[length(rows) + 1]] <- new_decision(type, paste0(prefix, "any"),
                                           p_min, truth_slope)
  rows[[length(rows) + 1]] <- new_decision(type, paste0(prefix, "any_bonf"),
    min(1, length(tails) * p_min), truth_slope)
  rows
}

# ---- One att_gt fit -> the pre-treatment cells and their covariance ----------
XF_ALL <- ~ X_1 + X_2 + X_3 + X_4 + X_5

fit_pre <- function(d, xformla = XF_ALL) {
  out <- tryCatch(
    att_gt(yname = "Y", tname = "time", idname = "unit_id",
           gname = "first_treat_period", xformla = xformla, data = d,
           est_method = doubleml_did_rf, control_group = "nevertreated",
           base_period = "universal", print_details = FALSE, pl = FALSE,
           cores = 1),
    error = function(e) NULL)
  if (is.null(out)) return(NULL)
  kk <- out$t - out$group
  sel <- which(kk < REF_K & is.finite(out$att) & is.finite(out$se) & out$se > 0)
  if (length(sel) < 2) return(NULL)
  inf <- as.matrix(out$inffunc)
  V <- (t(inf) %*% inf) / (out$n ^ 2)        # Cov(att), == V_analytical / n
  list(kk = as.integer(kk[sel]), att = out$att[sel], se = out$se[sel],
       V = V[sel, sel, drop = FALSE], Wpval = out$Wpval)
}

run_rep <- function(d) {
  tr <- truth_of(d); if (is.null(tr)) return(NULL)
  agg <- fit_pre(d); if (is.null(agg)) return(NULL)
  rows <- pre_rows(agg$kk, agg$att, agg$se, agg$V, tr$diff)
  # `did`'s own pre-test, halved onto the one-sided scale metrics.py reads.
  if (is.finite(agg$Wpval))
    rows[[length(rows) + 1]] <- new_decision("PRE", "joint",
                                             0.5 * agg$Wpval, tr$diff)
  rows
}

lin_folders <- c("linearity_degree=1", "linearity_degree=2", "linearity_degree=3")
options(warn = -1)
for (lin in lin_folders) {
  if (!dir.exists(lin)) { cat("no", lin, "- skip\n"); next }
  lin_degree <- as.integer(sub(".*=", "", lin))
  files <- list.files(lin, pattern = "^iteration_", full.names = TRUE)
  files <- files[grepl("csv$", files)]
  files <- files[order(as.integer(gsub("[^0-9]", "", basename(files))))]
  if (length(files) == 0) { cat("No files in", lin, "- skipping\n"); next }
  all_rows <- list()
  pb <- progress_bar$new(total = length(files), format = paste0(lin, " [:bar] :current/:total"))
  for (ii in seq_along(files)) {
    pb$tick(); rep <- ii - 1
    d <- read.csv(files[ii])
    d$first_treat_period[!is.finite(d$first_treat_period)] <- 0
    N <- length(unique(d$unit_id))
    rows <- tryCatch(run_rep(d), error = function(e) NULL)
    if (is.null(rows) || !length(rows)) next
    fr <- finalize(rows, rep, N, DGP, SETTING, lin_degree, METHOD)
    if (!is.null(fr)) all_rows[[length(all_rows) + 1]] <- fr
  }
  res <- if (length(all_rows)) do.call(rbind, all_rows) else data.frame()
  fn <- sprintf("summaries_%s_%s_lin_%d.csv", METHOD, SETTING, lin_degree)
  write.csv(res, fn, row.names = FALSE, na = ""); cat("\nwrote", fn, "(", nrow(res), "rows )\n")
}
sink()
"""
with open(f"{FOLDER}/DoubleML_pretrend.R", "w") as fh:
    fh.write(re.sub(r'SETTING <- "[^"]*"', f'SETTING <- "{SCENARIO}"',
                    DOUBLEML_PRETREND_R))
print("wrote", f"{FOLDER}/DoubleML_pretrend.R", "| SETTING =", SCENARIO)

In [ ]:
# ===== Run DoubleML's pre-trend test (loops the linearity degrees present) =====
import subprocess
if NUM_TREES != 500:
    subprocess.run(f"sed -i 's/num.trees = 500/num.trees = {NUM_TREES}/g' "
                   f"{FOLDER}/DoubleML_pretrend.R", shell=True, check=True)
subprocess.run(f"cd {FOLDER} && Rscript DoubleML_pretrend.R", shell=True, check=True)
print(open(f"{FOLDER}/output_DoubleML_pretrend.txt").read()[-3000:])

In [ ]:
# ===== Inspect + download (scored by the engine's own compute_metrics) =====
import os, glob, subprocess, pandas as pd
from IPython.display import display
from did_bcf_revision.metrics import compute_metrics

frames = []
for f in sorted(glob.glob(f"{FOLDER}/summaries_doubleml_{SCENARIO}_lin_*.csv")):
    frames.append(pd.read_csv(f)); print("loaded", os.path.basename(f))
# The DiD-BCF diagnostic's own summaries, if the clone carries them, so the two
# sit in one table.
for f in sorted(glob.glob(f"{ROOT}/Pretrend/summaries_pretrend_{SCENARIO}_lin_*.csv")):
    frames.append(pd.read_csv(f)); print("loaded DiD-BCF", os.path.basename(f))

if frames:
    summ = pd.concat(frames, ignore_index=True)
    met = compute_metrics(summ)
    pre = met[met["estimand_type"].isin(["PRE", "PRE_SUBC"])]
    cols = ["setting", "linearity_degree", "estimand_type", "estimand_id",
            "method", "n_reps", "mean_true", "bias", "cover95", "reject05", "role"]
    cols = [c for c in cols if c in pre.columns]
    print("\nPre-trend operating characteristics "
          "(reject05 = size where mean_true == 0, detection rate otherwise):")
    display(pre[cols].sort_values(["linearity_degree", "estimand_type",
                                   "estimand_id", "method"]))
    # The headline decision rules, side by side.
    head = pre[pre["estimand_id"].isin(["slope", "any_bonf", "joint",
                                        "X1_slope", "X1_any_bonf"])]
    if not head.empty:
        print("\nHeadline rules:")
        display(head.pivot_table(index=["linearity_degree", "estimand_id"],
                                 columns="method", values="reject05"))
else:
    print("No summaries written -- check the run cell output above.")

zipname = "DoubleML_{SCENARIO}_results.zip"
subprocess.run(f"cd {FOLDER} && zip -q {zipname} summaries_doubleml_*_lin_*.csv output_*.txt", shell=True)
try:
    from google.colab import files
    files.download(f"{FOLDER}/{zipname}")
except Exception as e:
    print("(not on Colab / download skipped):", e)